# Figure 10.2: Visualization of the running example points used throughout Section 10.2

Click a plotted point or use the point selector, then move it with sliders. The heatmap shows the currently selected proximity table.


In [ ]:
import math
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

DEFAULT_POINTS = [
    [0.0, 2.0],
    [2.0, 0.0],
    [3.0, 1.0],
    [5.0, 1.0],
]
METRIC_INFO = {
    'euclidean': {'table': 'Table 10.2', 'label': 'Euclidean', 'formula': 'sqrt((Δx)^2 + (Δy)^2)'},
    'manhattan': {'table': 'Table 10.3', 'label': 'Manhattan', 'formula': '|Δx| + |Δy|'},
    'linf': {'table': 'Table 10.4', 'label': 'L_infinity', 'formula': 'max(|Δx|, |Δy|)'},
    'l0': {'table': 'Table 10.5', 'label': 'L_0', 'formula': 'count of differing coordinates'},
}
points = [p[:] for p in DEFAULT_POINTS]
_syncing = False
current_fig = None


def fmt(value):
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    return f"{value:.3f}".rstrip('0').rstrip('.')


def pairwise(points_data, metric):
    n = len(points_data)
    mat = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            dx = abs(points_data[i][0] - points_data[j][0])
            dy = abs(points_data[i][1] - points_data[j][1])
            if metric == 'euclidean':
                value = math.hypot(dx, dy)
            elif metric == 'manhattan':
                value = dx + dy
            elif metric == 'linf':
                value = max(dx, dy)
            elif metric == 'l0':
                value = float((dx != 0) + (dy != 0))
            else:
                raise ValueError(metric)
            mat[i, j] = value
            mat[j, i] = value
    return mat


def closest_pair(matrix):
    best_value = None
    best_pair = (0, 1)
    n = matrix.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            value = matrix[i, j]
            if best_value is None or value < best_value:
                best_value = value
                best_pair = (i, j)
    return best_pair, best_value


def sync_sliders(*_):
    global _syncing
    if _syncing:
        return
    _syncing = True
    idx = point_select.index
    x_slider.value = points[idx][0]
    y_slider.value = points[idx][1]
    _syncing = False


def update_selected_point(change):
    global _syncing
    if _syncing:
        return
    idx = point_select.index
    points[idx][0] = float(x_slider.value)
    points[idx][1] = float(y_slider.value)
    render()


def on_point_click(trace, points_clicked, _):
    global _syncing
    if not points_clicked.point_inds:
        return
    idx = int(points_clicked.point_inds[0])
    _syncing = True
    point_select.value = f'x{idx + 1}'
    x_slider.value = points[idx][0]
    y_slider.value = points[idx][1]
    _syncing = False
    render()


def make_figure(metric, idx):
    matrix = pairwise(points, metric)
    nearest_pair, nearest_value = closest_pair(matrix)
    labels = [f'x{i}' for i in range(1, len(points) + 1)]
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    selected = points[idx]
    pair_pts = [points[nearest_pair[0]], points[nearest_pair[1]]]

    fig = go.FigureWidget(make_subplots(
        rows=1,
        cols=2,
        subplot_titles=('Figure 10.2: Running example points', f"{METRIC_INFO[metric]['table']}: {METRIC_INFO[metric]['label']} view"),
        specs=[[{'type': 'xy'}, {'type': 'heatmap'}]],
        horizontal_spacing=0.12,
    ))
    fig.add_trace(go.Scatter(
        x=[p[0] for p in pair_pts],
        y=[p[1] for p in pair_pts],
        mode='lines',
        line=dict(color='#f59e0b', width=3, dash='dash'),
        hoverinfo='skip',
        showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=xs,
        y=ys,
        mode='markers+text',
        text=labels,
        textposition='top center',
        marker=dict(size=12, color='#2563eb'),
        hovertemplate='%{text} = (%{x}, %{y})<extra></extra>',
        name='points',
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[selected[0]],
        y=[selected[1]],
        mode='markers',
        marker=dict(size=17, color='#f59e0b', line=dict(color='#92400e', width=2)),
        hovertemplate=f'selected: x{idx + 1}<extra></extra>',
        showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Heatmap(
        x=labels,
        y=labels,
        z=matrix,
        colorscale='Blues',
        zmin=0,
        zmax=max(float(matrix.max()), 1.0),
        showscale=False,
        hovertemplate='%{y} to %{x}: %{z:.3f}<extra></extra>',
    ), row=1, col=2)
    annotations = list(fig.layout.annotations)
    for r, row_values in enumerate(matrix):
        for c, value in enumerate(row_values):
            annotations.append(dict(x=labels[c], y=labels[r], xref='x2', yref='y2', text=fmt(value), showarrow=False, font=dict(size=12, color='#0f172a')))
    fig.update_layout(height=540, margin=dict(l=55, r=20, t=70, b=50), paper_bgcolor='#ffffff', plot_bgcolor='#ffffff')
    fig.update_layout(annotations=annotations)
    fig.update_xaxes(title='x_1', range=[min(xs) - 1.0, max(xs) + 1.0], gridcolor='#e2e8f0', zeroline=True, row=1, col=1)
    fig.update_yaxes(title='x_2', range=[min(ys) - 1.0, max(ys) + 1.0], gridcolor='#e2e8f0', zeroline=True, scaleanchor='x', scaleratio=1, row=1, col=1)
    fig.update_xaxes(side='top', row=1, col=2)
    fig.update_yaxes(autorange='reversed', row=1, col=2)
    fig.data[1].on_click(on_point_click)
    return fig, nearest_pair, nearest_value


def render(*_):
    global current_fig
    metric = metric_select.value
    idx = point_select.index
    current_fig, nearest_pair, nearest_value = make_figure(metric, idx)
    summary_lines = [
        f"Current view: {METRIC_INFO[metric]['table']} ({METRIC_INFO[metric]['label']})",
        f"Formula: {METRIC_INFO[metric]['formula']}",
        f"Closest pair: x{nearest_pair[0] + 1} and x{nearest_pair[1] + 1} with value {fmt(nearest_value)}",
        f"Selected point: x{idx + 1} = ({fmt(points[idx][0])}, {fmt(points[idx][1])})",
        '',
    ]
    for key in ['euclidean', 'manhattan', 'linf', 'l0']:
        pair_k, value_k = closest_pair(pairwise(points, key))
        summary_lines.append(f"{METRIC_INFO[key]['table']}: nearest pair is x{pair_k[0] + 1} and x{pair_k[1] + 1} ({fmt(value_k)})")
    summary_html.value = '<div style="white-space:pre-wrap;line-height:1.6;">' + '\n'.join(summary_lines) + '</div>'
    with plot_out:
        plot_out.clear_output(wait=True)
        display(current_fig)


point_select = widgets.ToggleButtons(options=['x1', 'x2', 'x3', 'x4'], description='point')
metric_select = widgets.ToggleButtons(
    options=[('Euclidean', 'euclidean'), ('Manhattan', 'manhattan'), ('L_infinity', 'linf'), ('L_0', 'l0')],
    description='metric'
)
x_slider = widgets.FloatSlider(value=points[0][0], min=-2.0, max=7.0, step=0.1, description='x', readout_format='.1f', continuous_update=False)
y_slider = widgets.FloatSlider(value=points[0][1], min=-2.0, max=4.0, step=0.1, description='y', readout_format='.1f', continuous_update=False)
reset_btn = widgets.Button(description='Reset default points')
plot_out = widgets.Output()
summary_html = widgets.HTML()

point_select.observe(sync_sliders, names='value')
metric_select.observe(render, names='value')
x_slider.observe(update_selected_point, names='value')
y_slider.observe(update_selected_point, names='value')


def on_reset(_):
    global points, _syncing
    points = [p[:] for p in DEFAULT_POINTS]
    _syncing = True
    point_select.value = 'x1'
    metric_select.value = 'euclidean'
    x_slider.value = points[0][0]
    y_slider.value = points[0][1]
    _syncing = False
    render()


reset_btn.on_click(on_reset)

controls = widgets.VBox([
    widgets.HTML(value='<h2 style="margin:0 0 8px;">Interactive Table 10.1 Data-Point Editor</h2><div style="color:#475569;margin-bottom:10px;">Click a plotted point or use the selector, move it with sliders, and view one derived table at a time as a heatmap.</div>'),
    point_select,
    widgets.HBox([x_slider, y_slider]),
    metric_select,
    reset_btn,
], layout=widgets.Layout(border='1px solid #cbd5e1', padding='12px', border_radius='14px'))

summary_card = widgets.VBox([
    widgets.HTML(value='<b>Metric Summary</b>'),
    summary_html,
], layout=widgets.Layout(border='1px solid #cbd5e1', padding='12px', border_radius='14px'))

render()
display(widgets.VBox([controls, plot_out, summary_card], layout=widgets.Layout(gap='16px')))
